TypeError: str expected, not NoneType

In [2]:
import os
import langchain
from dotenv import load_dotenv
load_dotenv()

os.environ["OPENAI-API_KEY"]=os.getenv("openai_api_key")
os.environ["GROQ_API_KEY"]=os.getenv("groq_api_key")
os.environ["GOOGLE_API_KEY"]=os.getenv("google_api_key")


In [3]:
from langchain.chat_models import init_chat_model

model = init_chat_model("groq:qwen/qwen3-32b")
response = model.invoke("Why do parrots talk?")
response

AIMessage(content='<think>\nOkay, so the user is asking, "Why do parrots talk?" Hmm, I need to figure out the best way to answer this. Let me start by recalling what I know about parrots and their ability to mimic human speech.\n\nFirst off, I remember that parrots are part of the Psittaciformes order, right? They\'re known for their intelligence and ability to imitate sounds. But why do they do that? Is it just to copy people, or is there a deeper reason?\n\nI think parrots are social animals. In the wild, they probably use vocalizations to communicate with each other. So when they\'re in captivity, they might mimic human speech as a way to interact with their human caregivers. That makes sense because they\'re trying to fit in and get attention. Like, if they make a sound that gets a positive response, they\'ll probably repeat it.\n\nAnother angle is their natural behavior. Parrots in the wild learn to mimic sounds to bond with their flock. Maybe when they\'re pets, they see humans a

In [4]:
from langchain.tools import tool

@tool
def get_weather(location:str)->str:
    """Get the weather at a location"""
    return f"It's sunny in {location}"


model_with_tools=model.bind_tools([get_weather])

In [5]:
response = model_with_tools.invoke("What's the weather like in Boston?")
print(response)
for tool_call in response.tool_calls:
    # View tool calls made by the model
    print(f"Tool: {tool_call['name']}")
    print(f"Args: {tool_call['args']}")

content='' additional_kwargs={'reasoning_content': 'Okay, the user is asking about the weather in Boston. I need to use the get_weather function. Let me check the function parameters. The required parameter is location, which should be a string. Boston is the location here. So I\'ll call the function with "Boston" as the location. Make sure the JSON is correctly formatted with the name and arguments.\n', 'tool_calls': [{'id': 'jtfcx8m15', 'function': {'arguments': '{"location":"Boston"}', 'name': 'get_weather'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 97, 'prompt_tokens': 154, 'total_tokens': 251, 'completion_time': 0.144672037, 'completion_tokens_details': {'reasoning_tokens': 73}, 'prompt_time': 0.008155535, 'prompt_tokens_details': None, 'queue_time': 0.051078004, 'total_time': 0.152827572}, 'model_name': 'qwen/qwen3-32b', 'system_fingerprint': 'fp_5cf921caa2', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_

In [6]:
# Step 1: Model generates tool calls
messages = [{"role": "user", "content": "What's the weather in Boston?"}]
ai_msg = model_with_tools.invoke(messages)
messages.append(ai_msg)

# Step 2: Execute tools and collect results
for tool_call in ai_msg.tool_calls:
    # Execute the tool with the generated arguments
    tool_result = get_weather.invoke(tool_call)
    messages.append(tool_result)

# Step 3: Pass results back to model for final response
final_response = model_with_tools.invoke(messages)
print(final_response.text)
# "The current weather in Boston is 72°F and sunny."

The weather in Boston is sunny.
